# SolarMap — scale-robust panel segmentationTwo stages:1. **Pretrain on BDAPPV with wide scale augmentation**, so the model works   across a range of ground sample distances instead of one.2. **Fine-tune on the Jbeil labels**, so it learns Lebanese rooftops.Produces `solar_unet.pt` for your local `models/` folder. **~2–2.5 hours** on aKaggle T4.---### Kaggle setup — all three, before you runRight-hand **Settings** panel:- **Accelerator → GPU T4 ×2**- **Internet → On** (off by default; the pip install and the 8 GB download both fail without it)Right-hand **Input** panel:- **+ Add Input → Datasets → Your Datasets → `jbeil06-seg`**Then **run cell 1 on its own**. It checks all three and prints `ready`, or tellsyou exactly what is missing. Fixing a setting there costs ten seconds; findingout an hour in costs the session.Once it says `ready`: **Save Version → Save & Run All (Commit)**. A committedrun executes on Kaggle's servers, so it survives your browser closing. A draftsession does not, and its `/kaggle/working` is wiped when it ends.*(Colab: Runtime ▸ Change runtime type ▸ T4 GPU. You are prompted to upload thezip at step 8.)*---### Why this notebook existsMeasured at Jbeil against 312 hand-verified arrays — same model, same roofs,same labels, only the imagery resampled:| gsd | arrays found | precision ||---|---|---|| 6.2 cm | 0.6% | 86.7% || 8.1 cm | 14.7% | 96.9% || **10.4 cm** | **23.2%** | **96.5%** || 12.5 cm | 11.4% | 93.1% || 15.0 cm | 6.2% | 90.6% || 20.0 cm | 5.6% | 82.4% |A 40× swing in recall from resolution alone, with a sharp peak. That is not aproperty of rooftops, it is a property of the old training pipeline:`RandomResizedCrop(scale=(0.7, 1.0))` varies linear scale by only ±16%, so themodel only ever saw arrays at one apparent size.Note what the table also says: **precision is already 96.5%** at the peak. Themodel is not confused, it is silent — it misses arrays rather than inventingthem. So the whole job here is recall.**Target to beat: 23.2% array recall at 96.5% precision.**

## 1 · Preflight — run this cell alone first

In [ ]:
import sys, os, socket, shutil
from pathlib import Path

# Kaggle and Colab differ in writable paths and in how files get in and out,
# so resolve that once here rather than sprinkling `if` statements later.
if Path("/kaggle").exists():
    ENV = "kaggle"
    WORK = Path("/kaggle/temp/solarmap")     # scratch: big, not preserved
    OUT  = Path("/kaggle/working")           # appears in the Output tab
elif "google.colab" in sys.modules or Path("/content").exists():
    ENV = "colab"
    WORK = OUT = Path("/content")
else:
    ENV = "local"
    WORK = OUT = Path("./work")
WORK.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
print(f"environment : {ENV}")
print(f"scratch     : {WORK}")
print(f"outputs     : {OUT}")

import torch
GPU = torch.cuda.is_available()
print("CUDA        :", GPU, "|", torch.cuda.get_device_name(0) if GPU else "NO GPU")
if not GPU:
    print("              -> Kaggle: Settings > Accelerator > GPU T4 x2")
    print("              -> Colab:  Runtime > Change runtime type > T4 GPU")

try:
    socket.create_connection(("zenodo.org", 443), timeout=8).close()
    NET = True; print("internet    : reachable")
except OSError:
    NET = False
    print("internet    : BLOCKED -> Kaggle Settings > Internet > On")

print(f"free disk   : {shutil.disk_usage(WORK).free/1e9:.0f} GB")

# Check the fine-tuning data is attached NOW, not at step 8. Discovering a
# missing dataset after the 8 GB download and two hours of training is the
# most expensive way to learn it, and a committed run cannot prompt you.
def find_dataset_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for q in base.rglob("train"):
        if (q/"images").is_dir():
            return q.parent
    return None

FT_ROOT = find_dataset_root("/kaggle/input") if ENV == "kaggle" else None
if ENV == "kaggle":
    if FT_ROOT:
        n = len(list((FT_ROOT/"train"/"images").glob("*")))
        print(f"fine-tune   : {FT_ROOT.name} ({n} train crops)")
    else:
        print("fine-tune   : NOT ATTACHED")
        print("              -> Input > + Add Input > Datasets > Your Datasets")
else:
    print("fine-tune   : will be uploaded at step 8")

ok = GPU and NET and (ENV != "kaggle" or FT_ROOT is not None)
print("")
print("ready -- Save Version > Save & Run All" if ok
      else "NOT READY -- fix the items above before committing a run")

## 2 · Dependencies

In [ ]:
!pip -q install segmentation-models-pytorch albumentations
import segmentation_models_pytorch as smp, albumentations as A
print("smp", smp.__version__, "| albumentations", A.__version__)

## 3 · Download BDAPPV`bdappv.zip` is ~8 GB — typically 10–20 min. The download resumes ifinterrupted, so rerunning the cell is safe.On Kaggle this goes to `/kaggle/temp`, not `/kaggle/working`: the workingdirectory has a 20 GB output cap that the zip plus its extraction would eat.

In [ ]:
import requests

REC = "7358126"
DEST = WORK / "bdappv.zip"

files_ = {f["key"]: f for f in requests.get(
    f"https://zenodo.org/api/records/{REC}").json()["files"]}
target = files_["bdappv.zip"]
url, size = target["links"]["self"], target["size"]
print(f"{size/1e9:.1f} GB from {url}")

done = DEST.stat().st_size if DEST.exists() else 0
if done < size:
    headers = {"Range": f"bytes={done}-"} if done else {}
    with requests.get(url, headers=headers, stream=True) as r, open(DEST, "ab") as fh:
        r.raise_for_status()
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
            done += len(chunk)
            if done % (256 << 20) < (1 << 20):
                print(f"  {done/1e9:.2f}/{size/1e9:.2f} GB", flush=True)
print("done:", DEST.stat().st_size)

In [ ]:
import zipfile

ROOT = WORK / "bdappv"
if not ROOT.exists():
    with zipfile.ZipFile(DEST) as z:
        z.extractall(WORK)
print("extracted. top level:", sorted(p.name for p in WORK.glob("bdappv*")))
for p in sorted(ROOT.rglob("*"))[:12]:
    if p.is_dir():
        print(f"  {p.relative_to(ROOT)}/  ({len(list(p.glob('*')))} entries)")

## 4 · Build the train/val splitBDAPPV ships unannotated images alongside the labelled ones. Those are kept asall-zero masks, capped at the number of positives — they are what teaches themodel that pools, skylights and dark flat roofs are *not* panels.

In [ ]:
import random
import numpy as np
from PIL import Image

SRC = ROOT / "google"
DATA = WORK / "dataset"
VAL_FRAC, SEED = 0.15, 1234

img_dir = next(SRC/d for d in ("img","images") if (SRC/d).is_dir())
mask_dir = next(SRC/d for d in ("mask","masks") if (SRC/d).is_dir())
masks = {p.stem: p for p in mask_dir.iterdir() if p.is_file()}
images = sorted(p for p in img_dir.iterdir() if p.is_file())

pos = [p for p in images if p.stem in masks]
neg = [p for p in images if p.stem not in masks]
random.Random(SEED).shuffle(neg)
neg = neg[:len(pos)]
print(f"{len(pos)} annotated, {len(neg)} negatives kept")

# Record the native pixel size -- the scale augmentation below is defined
# relative to it, so a different BDAPPV split would need this recomputed.
with Image.open(pos[0]) as _im:
    SRC_PX = _im.size[0]
print("source tile:", SRC_PX, "px over ~40 m =>", round(40/SRC_PX, 4), "m/px")

items = [(p, masks[p.stem]) for p in pos] + [(p, None) for p in neg]
random.Random(SEED).shuffle(items)
split = int(len(items) * (1 - VAL_FRAC))

for s in ("train/images","train/masks","val/images","val/masks"):
    (DATA/s).mkdir(parents=True, exist_ok=True)

blank = np.zeros((SRC_PX, SRC_PX), np.uint8)
for i, (ip, mp) in enumerate(items):
    sp = "train" if i < split else "val"
    Image.open(ip).convert("RGB").save(DATA/sp/"images"/f"{ip.stem}.png")
    if mp is None:
        Image.fromarray(blank).save(DATA/sp/"masks"/f"{ip.stem}.png")
    else:
        Image.open(mp).convert("L").save(DATA/sp/"masks"/f"{ip.stem}.png")
    if i % 5000 == 0:
        print(f"  {i}/{len(items)}", flush=True)
print("train/val:", len(list((DATA/'train/images').glob('*'))),
      len(list((DATA/'val/images').glob('*'))))

## 5 · Model, multi-scale pipeline and loss**This is the cell that differs from the old notebook.**BDAPPV google tiles are ~400 px over ~40 m, so ~0.10 m/px. Scaling an image bya factor `f` and then cropping `TILE` pixels yields an effective `0.10 / f`m/px. So `f ∈ [0.5, 2.0]` spans roughly **5–20 cm**, which brackets everyoperating point in the table above.`TILE` is 256 rather than 512 deliberately: at 256 the coarse end of that rangeneeds no padding, because a 400 px source scaled to `f = 0.64` is still 256 px.The network is fully convolutional, so it still runs on 512 px windows atinference — `config.yaml` controls that independently.Padding is `BORDER_REFLECT_101`, not zeros. A black border would teach an edgefeature that never occurs inside a real sliding window.

In [ ]:
import cv2, numpy as np, torch, torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# EPOCHS is 15, not 30. Stage A only has to teach general, scale-robust
# "what a rooftop array looks like" over ~38k images; the fine-tune on 312
# local arrays is what actually moves the number, and it is the cheap part.
# 30 epochs put a single run near 3 hours against a ~30 h/week GPU quota,
# which leaves no room for a second attempt.
ENCODER, TILE, BATCH, EPOCHS, LR = "resnet34", 256, 24, 15, 3e-4
SCALE_LO, SCALE_HI = 0.5, 2.0          # => ~0.05 to ~0.20 m/px from BDAPPV
WORKERS = 2 if ENV != "local" else 0

params = smp.encoders.get_preprocessing_params(ENCODER, "imagenet")
MEAN = tuple(float(v) for v in params["mean"])
STD  = tuple(float(v) for v in params["std"])

def build_tf(train, tile=TILE):
    if train:
        stages = [
            # The whole point of this notebook. The old value was equivalent
            # to f in [0.84, 1.0] -- a model that knows only one apparent size.
            A.RandomScale(scale_limit=(SCALE_LO - 1.0, SCALE_HI - 1.0), p=1.0),
            A.PadIfNeeded(min_height=tile, min_width=tile,
                          border_mode=cv2.BORDER_REFLECT_101, p=1.0),
            # Half the crops are steered onto the mask. Scaling UP to f=2 means
            # a 256 px window sees a quarter of the ground it used to, so a
            # plain random crop misses the array far more often: measured on
            # jbeil06_seg, the panel survived only 49.7% of draws against 70.3%
            # in the raw crops. A mask-aware crop restores that to 72.7%
            # without narrowing the scale range, and the random half still
            # supplies hard negatives.
            A.OneOf([
                A.CropNonEmptyMaskIfExists(height=tile, width=tile, p=1.0),
                A.RandomCrop(height=tile, width=tile, p=1.0),
            ], p=1.0),
            # Overhead imagery has no canonical orientation, so the full
            # dihedral group is valid augmentation rather than a distortion.
            A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=1.),
            # Mosaics stitch captures from different dates, sensors and sun
            # angles; this is the dominant domain shift after scale.
            A.RandomBrightnessContrast(.25,.25,p=.7),
            A.HueSaturationValue(10,20,12,p=.4),
            A.GaussNoise(p=.2), A.MotionBlur(blur_limit=3, p=.15),
        ]
    else:
        stages = [A.PadIfNeeded(min_height=tile, min_width=tile,
                                border_mode=cv2.BORDER_REFLECT_101, p=1.0),
                  A.CenterCrop(height=tile, width=tile, p=1.0)]
    return A.Compose(stages + [A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

class SegDS(Dataset):
    def __init__(self, root, split, train=None):
        self.imgs = sorted((Path(root)/split/"images").glob("*"))
        self.mdir = Path(root)/split/"masks"
        self.tf = build_tf(split == "train" if train is None else train)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        p = self.imgs[i]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        m = cv2.imread(str(self.mdir/p.name), cv2.IMREAD_GRAYSCALE)
        m = (np.zeros(img.shape[:2], np.uint8) if m is None else m)
        o = self.tf(image=img, mask=(m > 127).astype(np.float32))
        return o["image"], o["mask"].unsqueeze(0)

class DiceBCE(nn.Module):
    # Panels cover a small fraction of any tile. BCE alone scores well by
    # predicting "roof everywhere"; Dice makes it pay for missing positives.
    def __init__(self, w=.5):
        super().__init__(); self.w = w; self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, t):
        p = torch.sigmoid(logits)
        num = 2*(p*t).sum((1,2,3)) + 1
        den = p.sum((1,2,3)) + t.sum((1,2,3)) + 1
        return self.w*self.bce(logits, t) + (1-self.w)*(1-(num/den).mean())

print("mean", MEAN, "std", STD)
print(f"scale f in [{SCALE_LO}, {SCALE_HI}] => "
      f"{0.10/SCALE_HI:.3f} to {0.10/SCALE_LO:.3f} m/px from BDAPPV")

### Sanity check the augmentationConfirm the crops really do span a range of apparent sizes before spending twohours training. The same roof should appear at visibly different zooms.

In [ ]:
import matplotlib.pyplot as plt

_ds = SegDS(DATA, "train")
fig, ax = plt.subplots(2, 6, figsize=(16, 5.5))
for k in range(6):
    x, y = _ds[3]
    img = (x.numpy().transpose(1,2,0) * np.array(STD) + np.array(MEAN)).clip(0,1)
    ax[0,k].imshow(img); ax[0,k].axis("off")
    ax[1,k].imshow(y.numpy()[0], cmap="gray"); ax[1,k].axis("off")
ax[0,0].set_title("six draws of the SAME image", loc="left")
plt.tight_layout(); plt.show()

## 6 · Array-level scoringValidation IoU measures **pixels**. The project's target is *arrays located*,so score that instead — with the same matching rule as`scripts/score_arrays.py`, so these numbers predict the ones you get on thereal captures rather than merely correlating with them:- an **array is located** when ≥50% of its area is covered by prediction- a **detection is correct** when ≥50% of it lies on labelled panel

In [ ]:
def array_scores(model, loader, thr, dev, min_cover=.5, min_on=.5, max_batches=None):
    located = total = tp = fp = 0
    model.eval()
    with torch.no_grad():
        for bi, (x, y) in enumerate(loader):
            if max_batches and bi >= max_batches: break
            pr = (torch.sigmoid(model(x.to(dev))) > thr).cpu().numpy()[:, 0]
            gt = y.numpy()[:, 0] > .5
            for p, g in zip(pr, gt):
                n, lab = cv2.connectedComponents(g.astype(np.uint8))
                for i in range(1, n):
                    m = lab == i
                    if m.sum() < 4: continue
                    total += 1
                    located += bool((m & p).sum() / m.sum() >= min_cover)
                n2, lab2 = cv2.connectedComponents(p.astype(np.uint8))
                for i in range(1, n2):
                    m = lab2 == i
                    if m.sum() < 4: continue
                    if (m & g).sum() / m.sum() >= min_on: tp += 1
                    else: fp += 1
    # Plain Python floats, never numpy scalars. These end up in the checkpoint
    # as val_score, and torch.load defaults to weights_only=True from PyTorch
    # 2.6, which refuses to unpickle numpy._core.multiarray.scalar.
    rec = float(located / max(total, 1))
    prec = float(tp / max(tp + fp, 1))
    return rec, prec, float(2*prec*rec / max(prec+rec, 1e-9)), int(total)

def sweep(model, loader, dev, thrs=(.2,.3,.4,.5,.6,.7), max_batches=None):
    print(f"{'thr':>5} {'recall':>9} {'precision':>10} {'F1':>7}")
    rows = []
    for t in thrs:
        r, p, f, n = array_scores(model, loader, t, dev, max_batches=max_batches)
        rows.append((f, t, r, p)); print(f"{t:5.2f} {r:9.3f} {p:10.3f} {f:7.3f}")
    f, t, r, p = max(rows)
    print(f"best F1 {f:.3f} at threshold {t:.2f}  (recall {r:.3f}, precision {p:.3f})")
    return t

print("scoring helpers ready")

## 7 · Stage A — pretrain on BDAPPV

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tr = DataLoader(SegDS(DATA,"train"), BATCH, shuffle=True,
                num_workers=WORKERS, drop_last=True, pin_memory=True)
va = DataLoader(SegDS(DATA,"val"), BATCH, num_workers=WORKERS)
print(f"train={len(tr.dataset)} val={len(va.dataset)} device={dev}")

model = smp.Unet(ENCODER, encoder_weights="imagenet", in_channels=3, classes=1).to(dev)
crit = DiceBCE(.5)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler(dev.type, enabled=dev.type=="cuda")

def save_ckpt(path, score, ep, note):
    torch.save({"state_dict": model.state_dict(), "encoder": ENCODER,
                # Inference default; the net is fully convolutional so the
                # 256 px training window does not constrain it.
                "tile_size": 512, "train_tile": TILE,
                "scale_range": [float(SCALE_LO), float(SCALE_HI)],
                "mean": MEAN, "std": STD, "val_score": float(score),
                "epoch": int(ep), "stage": note}, path)

BDAPPV_CKPT = OUT / "solar_unet_bdappv.pt"
best = -1.0
for ep in range(1, EPOCHS+1):
    model.train(); run = 0.
    for x,y in tr:
        x,y = x.to(dev,non_blocking=True), y.to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(dev.type, enabled=dev.type=="cuda"):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run += loss.item()
    sched.step()

    model.eval(); inter = union = 0.
    with torch.no_grad():
        for x,y in va:
            x,y = x.to(dev), y.to(dev)
            p = (torch.sigmoid(model(x)) > .5).float()
            inter += (p*y).sum().item(); union += (p.sum()+y.sum()-(p*y).sum()).item()
    iou = float(inter/union) if union else 1.0
    flag = ""
    if iou > best:
        best = iou
        save_ckpt(BDAPPV_CKPT, iou, ep, "bdappv-multiscale")
        flag = "  <- saved"
    print(f"epoch {ep:3d}  loss={run/len(tr):.4f}  val_IoU={iou:.4f}{flag}", flush=True)
print(f"best val_IoU = {best:.4f}")

### Did the scale augmentation actually work?The point of stage A was scale robustness, so measure it. Each row re-rendersthe *same* validation roofs at a different apparent resolution. A model trainedthe old way collapses off its training scale; this one should stay roughly flat.

In [ ]:
class ScaledDS(SegDS):
    # Validation set rendered at a fixed scale factor, to probe robustness.
    def __init__(self, root, split, f):
        super().__init__(root, split, train=False)
        self.tf = A.Compose([
            A.Affine(scale=f, p=1.0),
            A.PadIfNeeded(min_height=TILE, min_width=TILE,
                          border_mode=cv2.BORDER_REFLECT_101, p=1.0),
            A.CenterCrop(height=TILE, width=TILE, p=1.0),
            A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

try:
    print(f"{'gsd m/px':>9} {'scale f':>8} {'recall':>9} {'precision':>10} {'F1':>7}")
    for f in (0.5, 0.7, 1.0, 1.4, 2.0):
        dl = DataLoader(ScaledDS(DATA, "val", f), BATCH, num_workers=WORKERS)
        r, p, s, n = array_scores(model, dl, 0.5, dev, max_batches=20)
        print(f"{0.10/f:9.3f} {f:8.2f} {r:9.3f} {p:10.3f} {s:7.3f}")
except Exception as e:
    # Diagnostic only -- never let it abort a committed run and cost the output.
    print("scale probe failed (non-fatal):", type(e).__name__, e)

## 8 · Stage B — fine-tune on the Jbeil labelsBDAPPV gives breadth: ~22,000 European rooftops. Your labels give localaccuracy: **312 hand-verified arrays at Jbeil**, re-drawn on 6 cm imagery wherea panel is 27 px across rather than 7.Build the dataset locally with:```powershellpython scripts/prepare_masks.py --capture jbeil-mb --name jbeil06_seg```then zip `data/datasets/jbeil06_seg` and add it as a Kaggle Dataset. Prefer it(806 crops) over `jbeil104_seg` (189) — scale augmentation covers the 10 cmoperating point from 6 cm data anyway, and 189 crops overfits in a few epochs.

In [ ]:
import zipfile

LOCAL_ROOT = None
if ENV == "kaggle":
    LOCAL_ROOT = FT_ROOT or find_dataset_root("/kaggle/input")
else:
    from google.colab import files
    print("Upload jbeil06_seg.zip ...")
    up = files.upload()
    with zipfile.ZipFile(list(up)[0]) as z:
        z.extractall(WORK/"local")
    LOCAL_ROOT = find_dataset_root(WORK/"local")

if LOCAL_ROOT is None:
    print("NO FINE-TUNING DATA -- stage B will be skipped and the BDAPPV")
    print("model shipped as-is. Attach the dataset and rerun to fix.")
else:
    print("dataset root:", LOCAL_ROOT)
    print("train:", len(list(Path(LOCAL_ROOT,'train/images').glob('*'))),
          "| val:", len(list(Path(LOCAL_ROOT,'val/images').glob('*'))))

In [ ]:
# Fine-tune from the BDAPPV weights at a tenth of the learning rate. A high LR
# would overwrite the general knowledge stage A just paid two hours to acquire
# -- the point is to nudge it toward Lebanon, not retrain it.
import shutil, traceback

FT_EPOCHS, FT_LR = 40, LR / 10
FINAL_CKPT = OUT / "solar_unet.pt"
best_f1 = -1.0
ft_va = None

# Everything below is wrapped. A committed run that raises here saves NOTHING,
# and losing two hours of GPU to a fixable error in the last stage is exactly
# what happened on the previous attempt. Whatever succeeded gets kept.
try:
    if LOCAL_ROOT is None:
        raise RuntimeError("no fine-tuning dataset attached")

    ft_tr = DataLoader(SegDS(LOCAL_ROOT, "train"), BATCH, shuffle=True,
                       num_workers=WORKERS, drop_last=True)
    ft_va = DataLoader(SegDS(LOCAL_ROOT, "val"), BATCH, num_workers=WORKERS)
    print(f"fine-tune on {len(ft_tr.dataset)} crops, validate on {len(ft_va.dataset)}")

    model.load_state_dict(torch.load(BDAPPV_CKPT, weights_only=False)["state_dict"])
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)

    # Selected on array F1, not IoU: IoU rewards tracing known arrays more
    # tightly, which is not the goal. Finding arrays it misses is.
    for ep in range(1, FT_EPOCHS+1):
        model.train()
        for x, y in ft_tr:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(dev.type, enabled=dev.type=="cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        r, p, f1, n = array_scores(model, ft_va, 0.5, dev)
        flag = ""
        if f1 > best_f1:
            best_f1 = f1
            save_ckpt(FINAL_CKPT, f1, ep, "jbeil-finetuned")
            flag = "  <- saved"
        print(f"ft {ep:3d}  recall={r:.3f} precision={p:.3f} F1={f1:.3f}{flag}",
              flush=True)

    print(f"best fine-tuned array F1 = {best_f1:.3f}")
except Exception as e:
    traceback.print_exc()
    print("STAGE B FAILED:", type(e).__name__, e)
    if not FINAL_CKPT.exists() and BDAPPV_CKPT.exists():
        shutil.copyfile(BDAPPV_CKPT, FINAL_CKPT)
        print("Shipped the BDAPPV model as solar_unet.pt so the run is not wasted.")

### Pick the operating pointYour target is **array recall**, and precision on the real captures is already~96%. So if a lower threshold buys recall at acceptable precision, take it —that trade is the whole game here.

In [ ]:
try:
    model.load_state_dict(torch.load(FINAL_CKPT, weights_only=False)["state_dict"])
    best_thr = sweep(model, ft_va, dev, thrs=(.1,.2,.3,.4,.5,.6,.7))
    print(f"Put this in config.yaml under inference.threshold: {best_thr}")
except Exception as e:
    # Advisory only. The threshold is better chosen against the real captures
    # anyway -- this val split is crops from the same 47 tiles.
    print("threshold sweep skipped (non-fatal):", type(e).__name__, e)

## 9 · Collect the checkpoint**Kaggle:** the files are in `/kaggle/working` — take them from the **Output**tab on the right.**Colab:** the cell below downloads them.Copy `solar_unet.pt` to `models/` in the repo, then verify against the *real*captures — the val split above is crops from the same 47 tiles, so it flattersthe model. The honest number comes from:```powershellpython scripts/detect.py --capture jbeil-mb-104 --checkpoint solar_unet.ptpython scripts/score_arrays.py --capture jbeil-mb-104 --labels labels_clean.json```Baseline to beat: **23.2% array recall at 96.5% precision.**

In [ ]:
print("files in", OUT)
for p in sorted(Path(OUT).glob("*")):
    if p.is_file():
        print(f"  {p.name:28} {p.stat().st_size/1e6:8.1f} MB")

if ENV == "colab":
    from google.colab import files
    for p in (FINAL_CKPT, BDAPPV_CKPT):
        if p.exists():
            files.download(str(p))
elif ENV == "kaggle":
    print("")
    print("Download from the Output panel on the right ->")